In [102]:
# imports

import pandas as pd
import numpy as np

import subprocess
import sys
import time
import json
import re
import requests
import os

# files and paths
tableFile = "tabela.xlsx"
criteriaFile = "criteria.txt"
OLLAMA_HOST = "http://localhost:11434"
OLLAMA_FOLDER = "ollama_models"
os.makedirs(OLLAMA_FOLDER, exist_ok=True)
os.environ["OLLAMA_MODELS"] = os.path.abspath(OLLAMA_FOLDER)  # store pulled models here

# Pull GGUF models directly from HuggingFace and run them with Ollama.
# Format: "hf.co/<user>/<repo>:<quantization>"
# Browse GGUF repos at https://huggingface.co/models?library=gguf
MODEL = "hf.co/unsloth/DeepSeek-R1-Distill-Llama-70B-GGUF:Q4_K_M"  # ~43GB
CATEGORIES = ["include", "exclude"] 

# output column for the AI veredict, named after the model so different
# models do not overwrite each other. Use a short tag (last path segment)
# so HuggingFace names like "hf.co/<user>/<repo>:<quant>" stay readable.
MODEL_TAG = MODEL.split("/")[-1]  # e.g. "Qwen2.5-7B-Instruct-GGUF:Q4_K_M"
SUGGESTION_COL = f"IA suggestion - {MODEL_TAG}"
COMMENT_COL = f"IA comment - {MODEL_TAG}"
COMMENT_MAXWORDS = 30  # max words in comment, to avoid long text in the table


In [103]:
os.system("ls")

README.md
bibFilter.py
bibParser.py
criteria.txt
filteringImage.def
filteringImage.sif
notas.md
ollama_models
planning.md
tabela.xlsx
tabela_safe.xlsx
tabela_with_IA_DeepSeek-R1-Distill-Llama-70B-GGUF:Q4_K_M.xlsx
tabela_with_IA_Qwen2.5-7B-Instruct-GGUF:Q4_K_M.xlsx
tableFilter.ipynb


0

In [104]:
table_df = pd.read_excel(tableFile)

relevant_fiels = ["title","abstract"]
display(table_df[relevant_fiels].head(10))

# add new columns for the AI veredict and its justification
# (column names include the model name):
table_df[SUGGESTION_COL] = ""
table_df[COMMENT_COL] = ""
display(table_df.head(10))


,title,abstract
0,Data-oriented strateg...,Full waveform inversion (FWI) of onshore targe...
1,1-D SEISMIC INVERSION OF DUAL WAVE-FIELD DATA ...,The inversion problem for determining seismic ...
2,1-D seismic inversion of dual wavefield data: ...,The inversion problem for determining seismic ...
3,12th International Congress for Applied Mechan...,The proceedings contain 26 papers. The special...
4,19th Formation Evaluation Symposium of Japan 2013,The proceedings contain 22 papers. The topics ...
5,1D Elastic Full-Waveform Inversion Through a R...,Summary We implement a transdimensional Bayesi...
6,2-D and 3-D Image-Domain Least-Squares Reverse...,The enormous computational overheads and exces...
7,2-D EM Scattering and Inverse Scattering From ...,This letter presents the computation of electr...
8,2.5-D Time-Domain Seismic Wavefield Modeling i...,Accurately modeling seismic wave propagation i...
9,2.5D forward and inverse modeling of elastic f...,We discuss two-and-half-dimensional (2.5D) for...


,bibtex_key,author,journal,year,source,pages,volume,document_type,doi,url,...,André,Hervé,Samuel,Raphael,Gabriel,Unnamed: 30,DECISÃO,CRITÈRIO DE EXCLUSÂO,IA suggestion - DeepSeek-R1-Distill-Llama-70B-GGUF:Q4_K_M,IA comment - DeepSeek-R1-Distill-Llama-70B-GGUF:Q4_K_M
0,NaN,"Trinh, P.T. and Brossier, R. and Métivier, L. ...",2018 SEG International Exposition and Annual M...,2019,Scopus,1213 - 1217,NaN,Conference paper,10.1190/segam2018-2997555.1,https://www.scopus.com/pages/publications/8505...,...,Não foca HPC,NaN,NaN,NaN,NaN,Não,NaN,NaN,,
1,NaN,"HILDEBRAND, ST and MCMECHAN, GA",GEOPHYSICS,1994,ISI Web of Science,782-788,59,Article,10.1190/1.1443636,NaN,...,Não é 3D,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,
2,NaN,"Hildebrand, S.T. and McMechan, G.A.",Geophysics,1994,Scopus,782 - 788,59,Article,10.1190/1.1443636,https://www.scopus.com/pages/publications/0028...,...,Não é 3D,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,
3,NaN,NaN,Lecture Notes in Mechanical Engineering,2025,Scopus,NaN,NaN,Conference review,NaN,https://www.scopus.com/pages/publications/1050...,...,Outros,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,
4,NaN,NaN,19th Formation Evaluation Symposium of Japan 2013,2013,Scopus,NaN,NaN,Conference review,NaN,https://www.scopus.com/pages/publications/8505...,...,Outros,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,
5,NaN,M. Aleardi and A. Salusti,Conference Proceedings,2019,Earthdoc,1,2019,NaN,https://doi.org/10.3997/2214-4609.201901339,https://www.earthdoc.org/content/papers/10.399...,...,Não é 3D,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,
6,NaN,"Zhang, Wei and Gao, Jinghuai",IEEE TRANSACTIONS ON GEOSCIENCE AND REMOTE SEN...,2022,ISI Web of Science,NaN,60,Article,10.1109/TGRS.2022.3215560,NaN,...,Não foca HPC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,
7,NaN,"Wu, Qing and Huang, Ruidong and Han, Feng",IEEE GEOSCIENCE AND REMOTE SENSING LETTERS,2024,ISI Web of Science,NaN,21,Article,10.1109/LGRS.2024.3378685,NaN,...,Não é 3D,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,
8,NaN,"Won, Moosoo and Zhou, Bing and Liu, Xu and Zem...",IEEE TRANSACTIONS ON GEOSCIENCE AND REMOTE SEN...,2023,ISI Web of Science,NaN,61,Article,10.1109/TGRS.2023.3333544,NaN,...,Foca só na modelagem,NaN,NaN,NaN,NaN,Avenir,NaN,NaN,,
9,NaN,"Xiong, J. and Abubakar, A. and Lin, Y. and Hab...",Society of Exploration Geophysicists Internati...,2011,Scopus,2395 - 2400,NaN,Conference paper,NaN,https://www.scopus.com/pages/publications/8505...,...,Duvida,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,


In [105]:
# Ollama related functions


# start the model

def start_ollama():
    proc = subprocess.Popen(
        ['ollama', 'serve'],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    # Wait for the HTTP server to come up (happens before GPU is ready)
    for _ in range(60):
        try:
            requests.get(f'{OLLAMA_HOST}/api/tags', timeout=1)
            return proc
        except Exception:
            time.sleep(1)
    sys.exit("Error: Ollama failed to start after 60 seconds.")


# list the models already downloaded locally
def list_models():
    """Return the names of models already available locally."""
    r = requests.get(f'{OLLAMA_HOST}/api/tags', timeout=10)
    r.raise_for_status()
    return [m['name'] for m in r.json().get('models', [])]


# download (pull) a model into the local Ollama store, showing progress
def pull_model(model):
    """Download an Ollama model, streaming progress. Skips if already present.
    Requires the Ollama server to be running (call start_ollama() first)."""
    if any(name == model or name.startswith(model + ':') for name in list_models()):
        print(f"{model} already downloaded.")
        return
    print(f"Pulling {model} ...")
    r = requests.post(
        f'{OLLAMA_HOST}/api/pull',
        json={'model': model, 'stream': True},
        stream=True,
        timeout=None,
    )
    r.raise_for_status()
    last_status = ''
    for line in r.iter_lines():
        if not line:
            continue
        msg = json.loads(line)
        if 'error' in msg:
            raise RuntimeError(f"Failed to pull {model}: {msg['error']}")
        status = msg.get('status', '')
        if 'total' in msg and 'completed' in msg and msg['total']:
            pct = msg['completed'] / msg['total'] * 100
            print(f"\r{status}: {pct:5.1f}%   ", end='', flush=True)
        elif status != last_status:
            print(f"\n{status}", end='', flush=True)
            last_status = status
    print("\nDone.")


# warmup, you need to wait the model to properlly start.
def warmup_model(model):
    """Send a tiny inference request to force the model to load into GPU memory.
    Retries until the model responds successfully or times out."""
    print(f"Loading model {model} into GPU memory...", file=sys.stderr)
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": "hi"}],
        "stream": False,
    }
    for attempt in range(1, 25):  # up to ~4 minutes (big models load slowly)
        try:
            r = requests.post(f'{OLLAMA_HOST}/api/chat', json=payload, timeout=120)
            if r.status_code == 200:
                print("Model ready.", file=sys.stderr)
                return
            print(f"  warmup attempt {attempt}: HTTP {r.status_code}, retrying...", file=sys.stderr)
        except Exception as e:
            print(f"  warmup attempt {attempt}: {e}, retrying...", file=sys.stderr)
        time.sleep(10)
    sys.exit("Error: model failed to load after warmup attempts.")


In [106]:

start_ollama()      # start the Ollama server (one call per kernel session)
print("Local models:", list_models())

pull_model(MODEL)   # downloads MODEL if not already present
warmup_model(MODEL) # load it into GPU memory, ready for IaSuggestion()


Local models: ['hf.co/unsloth/DeepSeek-R1-Distill-Llama-70B-GGUF:Q4_K_M', 'hf.co/bartowski/Qwen2.5-7B-Instruct-GGUF:Q4_K_M']
hf.co/unsloth/DeepSeek-R1-Distill-Llama-70B-GGUF:Q4_K_M already downloaded.


Loading model hf.co/unsloth/DeepSeek-R1-Distill-Llama-70B-GGUF:Q4_K_M into GPU memory...
Model ready.


In [107]:
# IA suggestion function

def IaSuggestion(text, criteria, veredict, model=None, maxwords=None):
    """Classifica um artigo em exatamente uma das categorias fornecidas.

    text     : str  -- título + resumo do artigo
    criteria : str  -- critérios de seleção
    veredict : list -- categorias permitidas (a categoria retornada é uma delas)
    returns  : tuple -- (category, reason): a categoria escolhida (um elemento de
                        `veredict`) e uma justificativa curta para a decisão
    """
    model = model or MODEL
    maxwords = maxwords or COMMENT_MAXWORDS
    options = "\n".join(f"- {v}" for v in veredict)
    system_msg = (
        "You classify papers for a systematic review / survey on High-Performance Computing (HPC) applied to seismic inversion (FWI/RTM).\n"
        f"\n{criteria}\n\n"
        "Classify the article in EXACTLY ONE of the following categories:\n"
        f"{options}\n\n"
        "Respond ONLY with JSON, without markdown and without extra text:\n"
        '{"category": "<one of the above categories, copied exactly>", '
        f'"reason": "<max {maxwords} words justifying the decision>"}}'
    )
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": text},
        ],
        "stream": False,
        "format": "json",  # force Ollama to return valid JSON
    }
    r = requests.post(f"{OLLAMA_HOST}/api/chat", json=payload, timeout=300)
    r.raise_for_status()
    raw = r.json()["message"]["content"].strip()

    # reasoning models (e.g. deepseek-r1) prepend a <think>...</think> block;
    # drop it so only the JSON answer remains
    raw = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL).strip()

    # parse the model's answer (category + reason)
    category = None
    reason = ""
    data = None
    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        if m:
            try:
                data = json.loads(m.group())
            except json.JSONDecodeError:
                pass
    if isinstance(data, dict):
        category = data.get("category")
        reason = data.get("reason", "") or ""

    # validate the answer against the allowed options (case-insensitive)
    if category:
        for v in veredict:
            if category.strip().lower() == v.strip().lower():
                return v, reason
    # fallback: pick the option whose text appears in the raw response
    # allow return None on problematic cases
    return None


In [108]:
## main loop on the table:

with open(criteriaFile, "r", encoding="utf-8") as f:
    criteria_str = f.read().strip()

MAX_entries = 100   # process at most this many rows; set to None to process all

processed = 0
for idx, row in table_df.iterrows():
    if MAX_entries is not None and processed >= MAX_entries:
        print(f"Reached MAX_entries = {MAX_entries}, stopping.")
        break

    title = "" if pd.isna(row["title"]) else str(row["title"])
    abstract = "" if pd.isna(row["abstract"]) else str(row["abstract"])
    if not title and not abstract:
        print(f"Row {idx}: skipped (no title/abstract)")
        continue

    text = f"{title}\n\n{abstract}"
    try:
        result = IaSuggestion(text, criteria_str, CATEGORIES)
        if result is None:
            # model output could not be parsed into a valid category
            suggestion, comment = "unparsed", ""
        else:
            suggestion, comment = result
        table_df.at[idx, SUGGESTION_COL] = suggestion
        table_df.at[idx, COMMENT_COL] = comment
        print(f"Row {idx}: {suggestion} — {comment}")
    except Exception as e:
        print(f"Row {idx}: Error during IA suggestion: {e}")

    processed += 1


Row 0: exclude — No computational or HPC aspects mentioned.
Row 1: exclude — The paper discusses 1-D seismic inversion, which is excluded as per the criteria.
Row 2: exclude — The abstract focuses on 1D seismic inversion methods without mentioning HPC or computational aspects.
Row 3: exclude — Abstract is a conference proceedings with multiple topics, none related to HPC or seismic inversion.
Row 4: exclude — The abstract is a table of contents without specific content about HPC or FWI/RTM.
Row 5: exclude — The paper focuses on 1D inversion, which is excluded by the criteria.
Row 6: exclude — The abstract does not mention any computational topics such as hardware, parallelization, or performance optimizations.
Row 7: exclude — Focuses on electromagnetic scattering and inversion in 2D, not FWI/RTM for seismic applications; lacks HPC content.
Row 8: include — Abstract mentions 2.5-D wave modeling relevant to FWI and discusses computational efficiency and parallelization.
Row 9: exclude —

In [109]:
# save the updated table to a new Excel file
output_file = f"tabela_with_IA_{MODEL_TAG}.xlsx"
table_df.to_excel(output_file, index=False)